### 1. Install Libraries: requests and pandas

In [16]:
pip install requests pandas numpy

Note: you may need to restart the kernel to use updated packages.


### 2. Run the India-Wide Data Grid Fetcher

In [17]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import pandas as pd
import numpy as np
import time

# --- 1. SET UP ROBUST CONNECTION HANDLER ---
# Use a Session to keep a single TCP connection alive
session = requests.Session()

# Configure automatic retries for network drops (e.g., Error 10053 or DNS failures)
retry_strategy = Retry(
    total=5,
    backoff_factor=1, 
    status_forcelist=[500, 502, 503, 504], # Note: 429 is intentionally removed here to handle it manually below
    allowed_methods=["GET"]
)
adapter = HTTPAdapter(max_retries=retry_strategy)
session.mount("https://", adapter)
session.mount("http://", adapter)

# --- 2. GRID & PARAMETERS ---
lat_min, lat_max = 8.0, 37.0
lon_min, lon_max = 68.0, 97.0
grid_step = 3.0 # Increase/Decrease this to change the density of the grid

lats = np.arange(lat_min, lat_max + 0.1, grid_step)
lons = np.arange(lon_min, lon_max + 0.1, grid_step)

api_features = [
    "temperature_2m", "relative_humidity_2m", "precipitation", 
    "dewpoint_2m", "surface_pressure", "cloud_cover", 
    "wind_speed_10m", "wind_direction_10m", "soil_temperature_0_to_7cm", 
    "soil_moisture_0_to_7cm", "soil_moisture_7_to_28cm", 
    "shortwave_radiation", "et0_fao_evapotranspiration", "vapour_pressure_deficit"
]

rename_dict = {
    'temperature_2m': 'temp_c',
    'relative_humidity_2m': 'humidity_pct',
    'precipitation': 'precip_mm',
    'dewpoint_2m': 'dewpoint_c',
    'surface_pressure': 'pressure_hpa',
    'cloud_cover': 'cloud_pct',
    'wind_speed_10m': 'wind_kmh',
    'wind_direction_10m': 'wind_dir_deg',
    'soil_temperature_0_to_7cm': 'soil_temp_shallow_c',
    'soil_moisture_0_to_7cm': 'soil_moist_shallow_pct',
    'soil_moisture_7_to_28cm': 'soil_moist_deep_pct',
    'shortwave_radiation': 'solar_wm2',
    'et0_fao_evapotranspiration': 'evap_mm',
    'vapour_pressure_deficit': 'vpd_kpa'
}

url = "https://archive-api.open-meteo.com/v1/archive"
all_data_frames = []

total_points = len(lats) * len(lons)
print(f"Fetching data for all of India across {total_points} grid locations...")

# --- 3. FETCH DATA WITH RATE LIMIT HANDLING ---
for lat in lats:
    for lon in lons:
        params = {
            "latitude": round(lat, 4),
            "longitude": round(lon, 4),
            "start_date": "2023-01-01",
            "end_date": "2023-12-31",
            "hourly": api_features,
            "timezone": "auto"
        }
        
        success = False
        
        while not success:
            try:
                response = session.get(url, params=params, timeout=15)
                
                if response.status_code == 200:
                    data = response.json()
                    df_point = pd.DataFrame(data['hourly'])
                    
                    df_point['latitude'] = round(lat, 4)
                    df_point['longitude'] = round(lon, 4)
                    
                    all_data_frames.append(df_point)
                    print(f"Success: ({lat}, {lon})")
                    success = True # Move to the next point
                    
                elif response.status_code == 429:
                    print(f"Rate limit hit (429) at ({lat}, {lon}). Cooling down for 60 seconds...")
                    time.sleep(60) 
                    
                else:
                    print(f"Skipping ({lat}, {lon}): HTTP {response.status_code}")
                    success = True # Break loop so it doesn't get stuck infinitely
                    
            except Exception as e:
                print(f"Network error at ({lat}, {lon}): {e}. Retrying in 10 seconds...")
                time.sleep(10)
                
        # Mandatory 1-second delay between successful requests
        time.sleep(1.0) 

print("\nAll points downloaded. Merging into a single master table...")

# --- 4. FORMAT AND SAVE ---
if len(all_data_frames) > 0:
    master_india_df = pd.concat(all_data_frames, ignore_index=True)
    master_india_df['time'] = pd.to_datetime(master_india_df['time'])
    master_india_df.rename(columns=rename_dict, inplace=True)
    
    ordered_cols = [
        'time', 'latitude', 'longitude', 'temp_c', 'humidity_pct', 'precip_mm', 
        'dewpoint_c', 'pressure_hpa', 'cloud_pct', 'wind_kmh', 'wind_dir_deg', 
        'soil_temp_shallow_c', 'soil_moist_shallow_pct', 'soil_moist_deep_pct', 
        'solar_wm2', 'evap_mm', 'vpd_kpa'
    ]
    master_india_df = master_india_df[ordered_cols]
    
    print("\nDataset Shape:", master_india_df.shape)
    master_india_df.to_csv("india_whole_country_weather_2023.csv", index=False)
    print("\nSuccessfully saved to 'india_whole_country_weather_2023.csv'!")
else:
    print("No data was fetched. Check your internet connection.")

Fetching data for all of India across 100 grid locations...
Success: (8.0, 68.0)
Success: (8.0, 71.0)
Success: (8.0, 74.0)
Success: (8.0, 77.0)
Success: (8.0, 80.0)
Success: (8.0, 83.0)
Success: (8.0, 86.0)
Success: (8.0, 89.0)
Success: (8.0, 92.0)
Success: (8.0, 95.0)
Success: (11.0, 68.0)
Success: (11.0, 71.0)
Success: (11.0, 74.0)
Success: (11.0, 77.0)
Success: (11.0, 80.0)
Success: (11.0, 83.0)
Success: (11.0, 86.0)
Success: (11.0, 89.0)
Success: (11.0, 92.0)
Success: (11.0, 95.0)
Success: (14.0, 68.0)
Success: (14.0, 71.0)
Success: (14.0, 74.0)
Success: (14.0, 77.0)
Success: (14.0, 80.0)
Success: (14.0, 83.0)
Success: (14.0, 86.0)
Success: (14.0, 89.0)
Success: (14.0, 92.0)
Success: (14.0, 95.0)
Success: (17.0, 68.0)
Success: (17.0, 71.0)
Success: (17.0, 74.0)
Success: (17.0, 77.0)
Success: (17.0, 80.0)
Success: (17.0, 83.0)
Success: (17.0, 86.0)
Success: (17.0, 89.0)
Success: (17.0, 92.0)
Success: (17.0, 95.0)
Success: (20.0, 68.0)
Success: (20.0, 71.0)
Success: (20.0, 74.0)
Succ

In [18]:
import os
# Define local path for Raw data
raw_path = r'E:\Apollo_AgriVerse\02_Datasets\Raw\weather'
os.makedirs(raw_path, exist_ok=True)

# Save the raw master dataset to local device with new name
raw_file_full = os.path.join(raw_path, 'weather_raw_dataset.csv')
master_india_df.to_csv(raw_file_full, index=False)
print(f"Raw dataset saved to: {raw_file_full}")

Raw dataset saved to: E:\Apollo_AgriVerse\02_Datasets\Raw\weather\weather_raw_dataset.csv


## 3. Cleaning Time Series Weather Data

### 3.1 Audit the Data Before Cleaning: Understand the Issues
First, let's check the basic information, missing values, and duplicated rows in the `master_india_df` before cleaning.

In [19]:
# Using the 'master_india_df' that was generated in the previous step
df = master_india_df.copy()

# 1. Check basic info (data types, number of non-null values)
print("Data Info:")
df.info()

# 2. Check for missing values in each column
print("\nMissing Values Count:")
print(df.isna().sum())

# 3. Check for duplicated rows
print(f"\nDuplicated Rows: {df.duplicated().sum()}")

Data Info:
<class 'pandas.DataFrame'>
RangeIndex: 876000 entries, 0 to 875999
Data columns (total 17 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   time                    876000 non-null  datetime64[us]
 1   latitude                876000 non-null  float64       
 2   longitude               876000 non-null  float64       
 3   temp_c                  876000 non-null  float64       
 4   humidity_pct            876000 non-null  int64         
 5   precip_mm               876000 non-null  float64       
 6   dewpoint_c              876000 non-null  float64       
 7   pressure_hpa            876000 non-null  float64       
 8   cloud_pct               876000 non-null  int64         
 9   wind_kmh                876000 non-null  float64       
 10  wind_dir_deg            876000 non-null  int64         
 11  soil_temp_shallow_c     876000 non-null  float64       
 12  soil_moist_shallow_pct  876000

### 3.2 Standardize the Time Index: Ensure Regular Intervals
We will convert the 'time' column to datetime objects, set it as the index, sort it, and then reindex to ensure an hourly frequency.

In [20]:
# 1. Ensure 'time' is a column and convert to datetime
if 'time' not in df.columns:
    df = df.reset_index()

df['time'] = pd.to_datetime(df['time'])

# 2. Set the time column as the index
df.set_index('time', inplace=True)

# 3. Sort the index to ensure chronological order
df.sort_index(inplace=True)

# 4. Reindex to a canonical hourly frequency ('h')
# The resample operation moves 'latitude' and 'longitude' into a MultiIndex.
# We reset the index first to avoid KeyError when dropping redundant columns.
df = df.groupby(['latitude', 'longitude']).resample('h').asfreq()
df = df.reset_index(level=[0, 1], drop=True).reset_index()

# Re-set 'time' as index after resample
df.set_index('time', inplace=True)

print("Time index standardized and resampled to hourly frequency.")
print(df.head())

Time index standardized and resampled to hourly frequency.
                     temp_c  humidity_pct  precip_mm  dewpoint_c  \
time                                                               
2023-01-01 00:00:00    27.5            74        0.0        22.5   
2023-01-01 01:00:00    27.5            73        0.0        22.2   
2023-01-01 02:00:00    27.5            72        0.0        22.1   
2023-01-01 03:00:00    27.3            73        0.0        22.1   
2023-01-01 04:00:00    27.2            73        0.0        22.1   

                     pressure_hpa  cloud_pct  wind_kmh  wind_dir_deg  \
time                                                                   
2023-01-01 00:00:00        1015.5         16      17.4            56   
2023-01-01 01:00:00        1015.0         10      17.9            56   
2023-01-01 02:00:00        1014.6          3      18.1            59   
2023-01-01 03:00:00        1014.2          7      17.7            57   
2023-01-01 04:00:00        1014.

### 3.3 Handle Missing Values (Imputation): Context-Aware Filling
Missing values will be handled using time-weighted interpolation for continuous signals and filling with 0 for precipitation.

In [21]:
import pandas as pd

continuous_cols = [
    'temp_c', 'humidity_pct', 'dewpoint_c', 'pressure_hpa', 'cloud_pct',
    'wind_kmh', 'wind_dir_deg', 'soil_temp_shallow_c', 'soil_moist_shallow_pct',
    'soil_moist_deep_pct', 'solar_wm2', 'evap_mm', 'vpd_kpa'
]

# 1. Force any index back into standard columns (fixes the KeyError if they were hidden)
df = df.reset_index()

# Drop the old index column if pandas created one during the reset
if 'index' in df.columns:
    df.drop(columns=['index'], inplace=True)

# 2. Ensure 'time' is a proper datetime object
if 'time' in df.columns:
    df['time'] = pd.to_datetime(df['time'])

# 3. DEFENSIVE SORT & INTERPOLATE
# Check if latitude and longitude actually exist in the dataframe
if 'latitude' in df.columns and 'longitude' in df.columns:
    
    # Sort by location and time
    df = df.sort_values(by=['latitude', 'longitude', 'time'])
    df.set_index('time', inplace=True)
    
    # Apply grouped interpolation
    for col in continuous_cols:
        if col in df.columns:
            df[col] = df.groupby(['latitude', 'longitude'], group_keys=False)[col].apply(
                lambda x: x.interpolate(method='time')
            )
else:
    print("Warning: 'latitude'/'longitude' not found. Assuming single-location dataset.")
    
    # Sort strictly by time
    df = df.sort_values(by=['time'])
    df.set_index('time', inplace=True)
    
    # Apply standard interpolation without groupby
    for col in continuous_cols:
        if col in df.columns:
            df[col] = df[col].interpolate(method='time')

# 4. Fill precipitation NaNs with 0 (No record = No rain)
if 'precip_mm' in df.columns:
    df['precip_mm'] = df['precip_mm'].fillna(0)

print("\nMissing values handled.")
print("\nRemaining Missing Values:")
print(df.isna().sum())


Missing values handled.

Remaining Missing Values:
temp_c                    0
humidity_pct              0
precip_mm                 0
dewpoint_c                0
pressure_hpa              0
cloud_pct                 0
wind_kmh                  0
wind_dir_deg              0
soil_temp_shallow_c       0
soil_moist_shallow_pct    0
soil_moist_deep_pct       0
solar_wm2                 0
evap_mm                   0
vpd_kpa                   0
dtype: int64


### 3.4 Detect and Handle Outliers: Identify Extreme Values
Outliers will be detected using the Interquartile Range (IQR) method for 'temp_c' and replaced with NaN, then re-interpolated.

In [22]:
def flag_outliers_iqr(dataframe, column_name):
    # Ensure the column is numeric before calculating quantiles
    dataframe[column_name] = pd.to_numeric(dataframe[column_name], errors='coerce')
    
    Q1 = dataframe[column_name].quantile(0.25)
    Q3 = dataframe[column_name].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Create a new column to flag outliers, keeping original values intact
    dataframe[f'{column_name}_is_outlier'] = ((dataframe[column_name] < lower_bound) | (dataframe[column_name] > upper_bound))
    return dataframe

# 1. Ensure all columns are out of the index for grouping
if df.index.name == 'time' or 'time' not in df.columns:
    df = df.reset_index()

# 2. Defensive check to ensure latitude/longitude exist
if 'latitude' in df.columns and 'longitude' in df.columns:
    # Apply to temperature column within each grid point using include_groups=False for modern pandas
    df = df.groupby(['latitude', 'longitude'], group_keys=False).apply(lambda group: flag_outliers_iqr(group, 'temp_c'))
else:
    # Fallback for single location datasets
    df = flag_outliers_iqr(df, 'temp_c')

# 3. Re-set 'time' as index
if 'time' in df.columns:
    df.set_index('time', inplace=True)

print("Outliers in 'temp_c' flagged in a new column 'temp_c_is_outlier'.")
print(df[['temp_c', 'temp_c_is_outlier']].head())

Outliers in 'temp_c' flagged in a new column 'temp_c_is_outlier'.
            temp_c  temp_c_is_outlier
time                                 
2023-01-01    27.5              False
2023-01-01    26.3              False
2023-01-01    26.0              False
2023-01-01   -13.6               True
2023-01-01    13.9              False


### 3.5 Smooth Noisy Data (Optional): Reduce High-Frequency Noise
An Exponentially Weighted Moving Average (EWMA) will be applied to the 'temp_c' column to smooth out noise.

In [23]:
import pandas as pd

# 1. Reset index to ensure all hidden levels (like time) become standard columns
df = df.reset_index()

# Clean up standard Pandas artifacts that might appear during a reset
for col in ['index', 'level_0']:
    if col in df.columns:
        df.drop(columns=[col], inplace=True)

# 2. Ensure temperature is numeric
if 'temp_c' in df.columns:
    df['temp_c'] = pd.to_numeric(df['temp_c'], errors='coerce')

# 3. Apply EWMA dynamically based on available columns
if 'latitude' in df.columns and 'longitude' in df.columns:
    # Scenario A: Dataset has multiple locations (The India Grid)
    df['temp_c_smoothed'] = df.groupby(['latitude', 'longitude'])['temp_c'].transform(
        lambda x: x.ewm(span=5, adjust=False).mean()
    )
    print("Temperature data smoothed using EWMA (grouped by location).")
else:
    # Scenario B: Dataset is a single location
    print("Note: 'latitude'/'longitude' columns not found. Applying EWMA directly without grouping.")
    df['temp_c_smoothed'] = df['temp_c'].ewm(span=5, adjust=False).mean()

# 4. Re-set 'time' as the index, as is best practice for time-series
if 'time' in df.columns:
    df.set_index('time', inplace=True)

# Display the results to confirm
print("\nPreview:")
print(df[['temp_c', 'temp_c_smoothed']].head())

Note: 'latitude'/'longitude' columns not found. Applying EWMA directly without grouping.

Preview:
            temp_c  temp_c_smoothed
time                               
2023-01-01    27.5        27.500000
2023-01-01    26.3        27.100000
2023-01-01    26.0        26.733333
2023-01-01   -13.6        13.288889
2023-01-01    13.9        13.492593


### 3.5.1 Data Consistency Validation
We need to ensure that values make physical sense. For instance, humidity should be between 0 and 100, and precipitation/wind speed should not be negative.

In [24]:
# Clip values to physical limits
df['humidity_pct'] = df['humidity_pct'].clip(0, 100)
df['precip_mm'] = df['precip_mm'].clip(lower=0)
df['wind_kmh'] = df['wind_kmh'].clip(lower=0)
df['cloud_pct'] = df['cloud_pct'].clip(0, 100)

print("Physical limit constraints applied to humidity, precipitation, wind speed, and cloud cover.")

Physical limit constraints applied to humidity, precipitation, wind speed, and cloud cover.


### 3.5.2 Temporal Feature Engineering
To help the model understand cyclic patterns, we'll extract temporal features like the hour of the day, month, and season.

In [25]:
df = df.reset_index()

# Extract temporal features
df['hour'] = df['time'].dt.hour
df['month'] = df['time'].dt.month
df['day_of_year'] = df['time'].dt.dayofyear

# Map months to seasons in India
# Winter: 1, 2 | Summer: 3, 4, 5, 6 | Monsoon: 7, 8, 9 | Post-Monsoon: 10, 11, 12
def get_india_season(month):
    if month in [1, 2]: return 'Winter'
    elif month in [3, 4, 5, 6]: return 'Summer'
    elif month in [7, 8, 9]: return 'Monsoon'
    else: return 'Post-Monsoon'

df['season'] = df['month'].apply(get_india_season)

df.set_index('time', inplace=True)
print("Temporal features (hour, month, day_of_year, season) added.")
display(df[['hour', 'month', 'season']].head())

Temporal features (hour, month, day_of_year, season) added.


,hour,month,season
time,,,
2023-01-01,0,1,Winter
2023-01-01,0,1,Winter
2023-01-01,0,1,Winter
2023-01-01,0,1,Winter
2023-01-01,0,1,Winter


### 3.6 Final Validation and Saving: Sanity Check
Finally, we will check for any remaining missing values, adjust data types, and save the cleaned dataset.

In [26]:
# Check if any missing values remain
print("\nRemaining Missing Values after all cleaning steps:")
print(df.isna().sum())

# Ensure data types are appropriate (e.g., downcasting floats to save memory)
float_cols = df.select_dtypes(include=['float64']).columns
df[float_cols] = df[float_cols].astype('float32')

# Save the cleaned dataset
df.to_csv("cleaned_india_whole_country_weather_2023.csv")
print("\nCleaned data saved successfully to 'cleaned_india_whole_country_weather_2023.csv'!")
print(df.info())
print(df.head())


Remaining Missing Values after all cleaning steps:
temp_c                    0
humidity_pct              0
precip_mm                 0
dewpoint_c                0
pressure_hpa              0
cloud_pct                 0
wind_kmh                  0
wind_dir_deg              0
soil_temp_shallow_c       0
soil_moist_shallow_pct    0
soil_moist_deep_pct       0
solar_wm2                 0
evap_mm                   0
vpd_kpa                   0
temp_c_is_outlier         0
temp_c_smoothed           0
hour                      0
month                     0
day_of_year               0
season                    0
dtype: int64

Cleaned data saved successfully to 'cleaned_india_whole_country_weather_2023.csv'!
<class 'pandas.DataFrame'>
DatetimeIndex: 876000 entries, 2023-01-01 00:00:00 to 2023-12-31 23:00:00
Data columns (total 20 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   temp_c                  876000 non-null  flo

In [27]:
import os
# Define local path for Interim (Cleaned) data
interim_path = r'E:\Apollo_AgriVerse\02_Datasets\Interim\weather'
os.makedirs(interim_path, exist_ok=True)

# Save the cleaned dataset to local device with new name
interim_file_full = os.path.join(interim_path, 'weather_cleaned_dataset.csv')
df.to_csv(interim_file_full)
print(f"Cleaned dataset saved to: {interim_file_full}")

Cleaned dataset saved to: E:\Apollo_AgriVerse\02_Datasets\Interim\weather\weather_cleaned_dataset.csv


## 4. Advanced ML Preprocessing Pipeline
This section enhances the dataset with cyclical encoding, lag features, rolling statistics, and robust scaling for machine learning readiness.

### 4.1 Memory Optimization & Cleaning
Ensuring chronological order and downcasting numeric types.

In [28]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler
import os

# --- 1. LOAD THE DATA ---
file_name = 'india_whole_country_weather_2023.csv'

# Check if file exists before trying to read it
if not os.path.exists(file_name):
    raise FileNotFoundError(f"Could not find '{file_name}'. Make sure the file is in the same folder as this notebook.")

df = pd.read_csv(file_name)
df_prep = df.copy()

# --- THE FIX: Convert 'time' to datetime objects immediately ---
if 'time' in df_prep.columns:
    df_prep['time'] = pd.to_datetime(df_prep['time'])
# ---------------------------------------------------------------

# 2. Safely reset index to ensure 'time', 'latitude', and 'longitude' are columns
if df_prep.index.name is not None or isinstance(df_prep.index, pd.MultiIndex):
    df_prep = df_prep.reset_index()

# Clean up any redundant 'index' columns
if 'index' in df_prep.columns:
    df_prep.drop(columns=['index'], inplace=True)

# 3. Ensure chronological order
sort_cols = [c for c in ['latitude', 'longitude', 'time'] if c in df_prep.columns]
df_prep.sort_values(by=sort_cols, inplace=True)

# 4. Replace extreme outliers (sensor errors) with NaN
df_prep.replace([-999, -9999], np.nan, inplace=True)

# 5. Time-based Interpolation per location
if 'latitude' in df_prep.columns and 'longitude' in df_prep.columns:
    df_prep = df_prep.groupby(['latitude', 'longitude'], group_keys=False).apply(
        lambda group: group.set_index('time').interpolate(method='time').ffill().bfill().reset_index()
    )

# 6. Downcast to save memory
float_cols = df_prep.select_dtypes(include=['float64', 'float32']).columns
df_prep[float_cols] = df_prep[float_cols].astype('float32')

print("Data cleaned and optimized for memory!")
print(f"Columns available: {df_prep.columns.tolist()}")

Data cleaned and optimized for memory!
Columns available: ['time', 'temp_c', 'humidity_pct', 'precip_mm', 'dewpoint_c', 'pressure_hpa', 'cloud_pct', 'wind_kmh', 'wind_dir_deg', 'soil_temp_shallow_c', 'soil_moist_shallow_pct', 'soil_moist_deep_pct', 'solar_wm2', 'evap_mm', 'vpd_kpa']


### 4.2 Cyclical Temporal Encoding
Encoding time cyclically (e.g., hour 23 is near hour 0).

In [29]:
# Extract basic time components
df_prep['hour'] = df_prep['time'].dt.hour
df_prep['month'] = df_prep['time'].dt.month
df_prep['day_of_year'] = df_prep['time'].dt.dayofyear

# Cyclical encoding for Hour (0-23)
df_prep['hour_sin'] = np.sin(2 * np.pi * df_prep['hour'] / 24.0)
df_prep['hour_cos'] = np.cos(2 * np.pi * df_prep['hour'] / 24.0)

# Cyclical encoding for Month (1-12)
df_prep['month_sin'] = np.sin(2 * np.pi * df_prep['month'] / 12.0)
df_prep['month_cos'] = np.cos(2 * np.pi * df_prep['month'] / 12.0)

# Drop linear hour/month as they are cyclically encoded
df_prep.drop(columns=['hour', 'month'], inplace=True)
print("Cyclical encoding complete.")

Cyclical encoding complete.


### 4.3 Lag Features & Rolling Statistics
Providing recent historical context and trend information.

In [30]:
# Check if spatial columns exist in the dataframe
has_spatial = 'latitude' in df_prep.columns and 'longitude' in df_prep.columns

if not has_spatial:
    print("Note: 'latitude'/'longitude' columns not found. Applying features directly without grouping.")

# 1. Create Lag Features (1h, 3h, 24h)
lag_targets = ['temp_c', 'humidity_pct', 'precip_mm', 'pressure_hpa']
for col in lag_targets:
    if has_spatial:
        df_prep[f'{col}_lag_1h'] = df_prep.groupby(['latitude', 'longitude'])[col].shift(1)
        df_prep[f'{col}_lag_3h'] = df_prep.groupby(['latitude', 'longitude'])[col].shift(3)
        df_prep[f'{col}_lag_24h'] = df_prep.groupby(['latitude', 'longitude'])[col].shift(24)
    else:
        df_prep[f'{col}_lag_1h'] = df_prep[col].shift(1)
        df_prep[f'{col}_lag_3h'] = df_prep[col].shift(3)
        df_prep[f'{col}_lag_24h'] = df_prep[col].shift(24)

# 2. Rolling Window Statistics (6h mean and std)
for col in ['temp_c', 'humidity_pct']:
    if has_spatial:
        df_prep[f'{col}_roll_mean_6h'] = df_prep.groupby(['latitude', 'longitude'])[col].transform(
            lambda x: x.rolling(window=6, min_periods=1).mean()
        )
        df_prep[f'{col}_roll_std_6h'] = df_prep.groupby(['latitude', 'longitude'])[col].transform(
            lambda x: x.rolling(window=6, min_periods=2).std()
        ).fillna(0)
    else:
        df_prep[f'{col}_roll_mean_6h'] = df_prep[col].rolling(window=6, min_periods=1).mean()
        df_prep[f'{col}_roll_std_6h'] = df_prep[col].rolling(window=6, min_periods=2).std().fillna(0)

# Drop rows with NaNs from shifts (this will drop the first 24 hours of data)
df_prep.dropna(inplace=True)
print("Lag and Rolling features created successfully!")

Note: 'latitude'/'longitude' columns not found. Applying features directly without grouping.
Lag and Rolling features created successfully!


### 4.4 Feature Scaling (Robust)
Scaling numerical columns while preserving extreme weather outliers.

In [31]:
# Identify columns to scale (exclude IDs, time, and categorical/flag columns)
exclude_cols = ['time', 'latitude', 'longitude', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'day_of_year', 'temp_c_is_outlier', 'season']
scale_cols = [col for col in df_prep.columns if col not in exclude_cols and df_prep[col].dtype != 'object']

# Scale using RobustScaler
scaler = RobustScaler()
df_prep[scale_cols] = scaler.fit_transform(df_prep[scale_cols])

print("\n--- Final Preprocessed Dataset ---")
display(df_prep.head())

# Save to CSV
df_prep.to_csv("india_weather_preprocessed_ready.csv", index=False)
print(f"\nSuccessfully saved preprocessed data! Shape: {df_prep.shape}")


--- Final Preprocessed Dataset ---


,time,temp_c,humidity_pct,precip_mm,dewpoint_c,pressure_hpa,cloud_pct,wind_kmh,wind_dir_deg,soil_temp_shallow_c,...,precip_mm_lag_1h,precip_mm_lag_3h,precip_mm_lag_24h,pressure_hpa_lag_1h,pressure_hpa_lag_3h,pressure_hpa_lag_24h,temp_c_roll_mean_6h,temp_c_roll_std_6h,humidity_pct_roll_mean_6h,humidity_pct_roll_std_6h
24,2023-01-02 00:00:00,0.098361,-0.107143,0.0,0.100592,0.270270,-0.427083,1.138211,-0.859459,0.059829,...,0.0,0.0,0.0,0.274775,0.277778,0.283784,0.112813,-0.446901,-0.10000,-0.492093
25,2023-01-02 01:00:00,0.090164,-0.107143,0.0,0.094675,0.264265,-0.114583,1.016260,-0.859459,0.051282,...,0.0,0.0,0.0,0.270270,0.279280,0.276276,0.107242,-0.421662,-0.10000,-0.492093
26,2023-01-02 02:00:00,0.081967,-0.107143,0.0,0.088757,0.256757,0.322917,0.926829,-0.864865,0.051282,...,0.0,0.0,0.0,0.264265,0.274775,0.270270,0.100279,-0.405354,-0.10625,-0.537551
27,2023-01-02 03:00:00,0.057377,-0.035714,0.0,0.100592,0.249250,-0.291667,0.780488,-0.832432,0.051282,...,0.0,0.0,0.0,0.256757,0.270270,0.264265,0.089136,-0.363359,-0.08750,-0.447780
28,2023-01-02 04:00:00,0.057377,-0.071429,0.0,0.076923,0.246246,-0.052083,0.682927,-0.805405,0.051282,...,0.0,0.0,0.0,0.249250,0.264265,0.262763,0.079387,-0.360168,-0.07500,-0.475960



Successfully saved preprocessed data! Shape: (875976, 36)


### 4.5 Additional Meteorological Feature Engineering
We will add wind vectors, saturation proxies, and extreme event indicators.

In [32]:
import numpy as np

# 1. Decompose Wind into U and V components (Vectorization)
# Convert degrees to radians
wind_rad = np.deg2rad(df_prep['wind_dir_deg'])
df_prep['wind_u'] = df_prep['wind_kmh'] * np.cos(wind_rad)
df_prep['wind_v'] = df_prep['wind_kmh'] * np.sin(wind_rad)

# 2. Dew Point Depression (Temperature - Dew Point)
df_prep['dew_point_depression'] = df_prep['temp_c'] - df_prep['dewpoint_c']

# 3. Extreme Event Flagging (Binary Indicators)
# Heatwave threshold (>40C) and Heavy Rain threshold (>10mm/h)
df_prep['is_extreme_heat'] = (df_prep['temp_c'] > 40).astype(int)
df_prep['is_heavy_precip'] = (df_prep['precip_mm'] > 10).astype(int)

# 4. Temperature Delta (Change from previous hour)
# Check if spatial columns exist before grouping
if 'latitude' in df_prep.columns and 'longitude' in df_prep.columns:
    df_prep['temp_delta_1h'] = df_prep.groupby(['latitude', 'longitude'])['temp_c'].diff().fillna(0)
else:
    # Calculate difference directly for a single location
    df_prep['temp_delta_1h'] = df_prep['temp_c'].diff().fillna(0)

print("Additional physical and extreme event features added.")

Additional physical and extreme event features added.


### 4.6 Final Pipeline Validation
A final look at the feature set before it is finalized for training.

In [33]:
# Update scaling for the new numeric features
new_numeric_cols = ['wind_u', 'wind_v', 'dew_point_depression', 'temp_delta_1h']
df_prep[new_numeric_cols] = scaler.fit_transform(df_prep[new_numeric_cols])

print(f"Final Feature count: {len(df_prep.columns)}")
print(df_prep.columns.tolist())
display(df_prep.tail())

# Save the final augmented dataset
df_prep.to_csv("india_weather_augmented_final.csv", index=False)
print("Augmented dataset saved as 'india_weather_augmented_final.csv'")

Final Feature count: 42
['time', 'temp_c', 'humidity_pct', 'precip_mm', 'dewpoint_c', 'pressure_hpa', 'cloud_pct', 'wind_kmh', 'wind_dir_deg', 'soil_temp_shallow_c', 'soil_moist_shallow_pct', 'soil_moist_deep_pct', 'solar_wm2', 'evap_mm', 'vpd_kpa', 'day_of_year', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'temp_c_lag_1h', 'temp_c_lag_3h', 'temp_c_lag_24h', 'humidity_pct_lag_1h', 'humidity_pct_lag_3h', 'humidity_pct_lag_24h', 'precip_mm_lag_1h', 'precip_mm_lag_3h', 'precip_mm_lag_24h', 'pressure_hpa_lag_1h', 'pressure_hpa_lag_3h', 'pressure_hpa_lag_24h', 'temp_c_roll_mean_6h', 'temp_c_roll_std_6h', 'humidity_pct_roll_mean_6h', 'humidity_pct_roll_std_6h', 'wind_u', 'wind_v', 'dew_point_depression', 'is_extreme_heat', 'is_heavy_precip', 'temp_delta_1h']


,time,temp_c,humidity_pct,precip_mm,dewpoint_c,pressure_hpa,cloud_pct,wind_kmh,wind_dir_deg,soil_temp_shallow_c,...,temp_c_roll_mean_6h,temp_c_roll_std_6h,humidity_pct_roll_mean_6h,humidity_pct_roll_std_6h,wind_u,wind_v,dew_point_depression,is_extreme_heat,is_heavy_precip,temp_delta_1h
8755,2023-12-31 19:00:00,-2.852459,-2.178571,0.0,-3.071006,-6.343840,-0.020833,-0.195122,0.324324,-3.307692,...,-2.816156,0.185195,-2.25000,-0.257074,-0.195122,-0.300829,0.924402,0,0,0.000000
8756,2023-12-31 20:00:00,-2.991803,-2.107143,0.0,-3.071006,-6.363360,0.458333,-0.317073,-0.994595,-3.418803,...,-2.849582,0.548582,-2.26250,-0.381247,-0.317030,0.887547,0.525153,0,0,-2.428570
8757,2023-12-31 21:00:00,-3.409836,-1.714286,0.0,-2.994083,-6.445942,0.031250,-0.089431,-0.875676,-3.478632,...,-2.969359,1.761701,-2.18750,0.412373,-0.089422,0.143579,-0.892996,0,0,-7.285712
8758,2023-12-31 22:00:00,-3.385246,-1.714286,0.0,-2.982248,-6.418915,0.458333,-0.113821,-0.940541,-3.529914,...,-3.091922,1.996084,-2.10625,0.676882,-0.113808,0.233781,-0.856448,0,0,0.428572
8759,2023-12-31 23:00:00,-3.516393,-1.357143,0.0,-2.857988,-6.391888,0.177083,0.227642,-0.891892,-3.555555,...,-3.233983,1.996084,-1.95000,1.162056,0.227618,-0.739430,-1.588243,0,0,-2.285713


Augmented dataset saved as 'india_weather_augmented_final.csv'


In [34]:
import os
# Define local path for Processed data
processed_path = r'E:\Apollo_AgriVerse\02_Datasets\Processed\weather'
os.makedirs(processed_path, exist_ok=True)

# Save the final processed dataset to local device with new name
processed_file_full = os.path.join(processed_path, 'weather_processed_dataset.csv')
df_prep.to_csv(processed_file_full, index=False)
print(f"Processed dataset saved to: {processed_file_full}")

Processed dataset saved to: E:\Apollo_AgriVerse\02_Datasets\Processed\weather\weather_processed_dataset.csv
